# 15.11 推荐系统评估 / Evaluating Recommenders

**中文**：这是 Part 15 的收官，也是面试里**最容易被追问、最能区分高手与新手**的一节：*"你怎么证明你的推荐系统更好？"* 前面每一节我们都和"热门基线"比 Precision/Recall，但推荐评估远不止这两个指标。本节系统讲清**排序指标（precision@k / recall@k / HR / MAP / MRR / NDCG）**、**超越准确率的指标（覆盖率/新颖性/多样性）**，以及两个最常见的**评估陷阱**。
**English**: This is the finale of Part 15 and the most interview-probed, novice-vs-expert-distinguishing topic: *"How do you prove your recommender is better?"* Every prior section compared Precision/Recall against a popularity baseline, but recsys evaluation is far richer. This section systematically covers **ranking metrics (precision@k / recall@k / HR / MAP / MRR / NDCG)**, **beyond-accuracy metrics (coverage / novelty / diversity)**, and the two most common **evaluation pitfalls**.

---

**中文**：推荐的本质是**给每个用户输出一个有序的 Top-K 列表**，所以评估关心的是**排序质量**：相关的物品有没有排在前面。所有指标都基于两样东西：① 模型给用户的 Top-K 推荐列表；② 该用户的"相关物品"集合（held-out 测试集中他真正喜欢的）。
**English**: Recommendation outputs an **ordered Top-K list per user**, so evaluation is about **ranking quality**: are relevant items near the top? Every metric is built from two things: ① the model's Top-K list for a user; ② that user's "relevant items" set (those they truly liked in the held-out test set).

| 指标 / Metric | 公式直觉 / Intuition | 是否看顺序 / Order-aware? |
|---|---|---|
| **Precision@K** | 前K个里有多少是相关的 / fraction of top-K that are relevant | 否 / No |
| **Recall@K** | 所有相关物品里有多少被召回到前K / fraction of relevant captured in top-K | 否 / No |
| **Hit Rate@K** | 前K里**至少有一个**相关就算命中 / at least one relevant in top-K | 否 / No |
| **MRR** | 第一个相关物品排名的倒数 / reciprocal rank of the first hit | 是 / Yes |
| **MAP** | 每个相关物品位置上 precision 的平均 / mean of precision at each hit | 是 / Yes |
| **NDCG@K** | 带位置折扣的增益 / log-discounted gain, normalized | 是 / Yes |

> 💡 **面试速查 / Interview cheat-sheet（★★★ 必考）**
> **中文**：**Precision/Recall/HR 不看排序**（前K是个集合）；**MRR/MAP/NDCG 看排序**（越靠前奖励越高）。**NDCG 最常用**——$DCG@K=\sum \frac{rel_i}{\log_2(i+1)}$，除以理想排序的 IDCG 归一化到 [0,1]，支持多档相关度。**两大陷阱**：① **采样评估**（只对少量负样本排序）会**严重高估且可能颠倒模型排名**（Krichene & Rendle 2020），要用**全量排序**；② **离线≠在线**——离线指标受日志偏差影响，最终要 A/B。**别只看准确率**：覆盖率/多样性/新颖性同样重要（只推热门 precision 不低但体验差）。
> **English**: **Precision/Recall/HR ignore order** (top-K is a set); **MRR/MAP/NDCG are order-aware** (higher rank = more reward). **NDCG is the most used** — $DCG@K=\sum \frac{rel_i}{\log_2(i+1)}$, divided by the ideal IDCG to normalize to [0,1], supports graded relevance. **Two pitfalls**: ① **sampled evaluation** (ranking against a few negatives) **massively inflates and can even reorder models** (Krichene & Rendle 2020) — use **full ranking**; ② **offline ≠ online** — offline metrics suffer log bias; ultimately A/B test. **Don't only look at accuracy**: coverage/diversity/novelty matter too (recommending only blockbusters has decent precision but poor experience).


In [ ]:

# ============================================================
# 数据 + 三个模型(热门/物品CF/BPR) / data + three models to compare
# ============================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(0)
R=os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
rat=pd.read_csv(os.path.join(R,"u.data"),sep="\t",names=["user","item","rating","ts"])
GEN=["unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary","Drama",
     "Fantasy","FilmNoir","Horror","Musical","Mystery","Romance","SciFi","Thriller","War","Western"]
mv=pd.read_csv(os.path.join(R,"u.item"),sep="|",encoding="latin-1",header=None,
               names=["item","title","date","v","url"]+GEN).set_index("item")
rs=rat.sort_values("ts"); a=[];b=[]
for _,g in rs.groupby("user"):
    c=int(len(g)*0.8); a.append(g.iloc[:c]); b.append(g.iloc[c:])
train=pd.concat(a); test=pd.concat(b)
uids=np.sort(rat["user"].unique()); iids=np.sort(rat["item"].unique())
u2x={u:i for i,u in enumerate(uids)}; i2x={i:j for j,i in enumerate(iids)}; m,n=len(uids),len(iids)
def pos(df):
    d=df[df["rating"]>=4]; return np.array([u2x[u] for u in d["user"]]),np.array([i2x[i] for i in d["item"]])
trU,trI=pos(train); teU,teI=pos(test)
seen=[set() for _ in range(m)]
for u,i in zip(trU,trI): seen[u].add(i)
test_pos=[set() for _ in range(m)]
for u,i in zip(teU,teI):
    if i not in seen[u]: test_pos[u].add(i)
pop=np.bincount(trI,minlength=n).astype(float)
genre=np.stack([mv.loc[i,GEN].values.astype(float) for i in iids])           # 物品类型向量(算多样性用)/ genre vectors
gnorm=genre/np.maximum(np.linalg.norm(genre,axis=1,keepdims=True),1e-9)

# 物品-物品 CF（中心化余弦, 用 sum-of-sims 排序）/ item-item CF
Rm=np.zeros((m,n))
for u,i,r in zip(train["user"].map(u2x),train["item"].map(i2x),train["rating"]): Rm[u,i]=r
mk=Rm>0; um=np.array([Rm[u,mk[u]].mean() if mk[u].any() else 0 for u in range(m)])
Rc=np.where(mk,Rm-um[:,None],0); In=Rc/np.maximum(np.linalg.norm(Rc,axis=0,keepdims=True),1e-9); Sii=In.T@In
liked=[np.where(Rm[u]>=4)[0] for u in range(m)]

# BPR（隐式 MF）/ BPR
def train_bpr(k=32,lr=0.05,reg=0.01,ep=15):
    rng=np.random.default_rng(0); P=rng.normal(0,0.1,(m,k)); Q=rng.normal(0,0.1,(n,k)); bi=np.zeros(n)
    for e in range(ep):
        for t in rng.permutation(len(trU)):
            u=trU[t]; i=trI[t]; j=rng.integers(n)
            while j in seen[u]: j=rng.integers(n)
            s=1/(1+np.exp(bi[i]-bi[j]+P[u]@(Q[i]-Q[j]))); Pu=P[u].copy()
            P[u]+=lr*(s*(Q[i]-Q[j])-reg*P[u]); Q[i]+=lr*(s*Pu-reg*Q[i]); Q[j]+=lr*(-s*Pu-reg*Q[j])
            bi[i]+=lr*(s-reg*bi[i]); bi[j]+=lr*(-s-reg*bi[j])
    return P,Q,bi
P,Q,bi=train_bpr()
score = {
 "Popularity": lambda u: pop.copy(),
 "ItemCF":     lambda u: (Sii[:,liked[u]].sum(1) if len(liked[u]) else pop.copy()),
 "BPR":        lambda u: bi+Q@P[u],
}
print("模型就绪 / models ready:", list(score)); print("用户", m, "物品", n)


**中文**：现在**从零实现全部指标**。对每个用户：取模型的 Top-K 列表（排除训练已见），与他的相关集合比对。注意 **NDCG** 的关键——它用 $\frac{1}{\log_2(i+1)}$ 给靠前的命中更高权重，再除以"理想排序"的 IDCG 做归一化，所以同样命中 1 个，排第 1 比排第 10 得分高得多。
**English**: Now **implement every metric from scratch**. For each user: take the model's Top-K list (excluding training-seen items) and compare to their relevant set. The key for **NDCG** — it weights earlier hits by $\frac{1}{\log_2(i+1)}$ and divides by the ideal IDCG, so the same single hit scores far higher at rank 1 than at rank 10.


In [ ]:

# ============================================================
# 从零实现所有评估指标 / all ranking metrics from scratch
# ============================================================
def topk(fn,u,K):
    sc=fn(u).copy()
    for i in seen[u]: sc[i]=-1e18                              # 屏蔽训练已见 / mask seen
    return np.argsort(sc)[::-1][:K]                            # 取分数最高的前K(有序) / ordered top-K

def evaluate(fn, K=10):
    acc=dict(Precision=0,Recall=0,HitRate=0,MAP=0,MRR=0,NDCG=0,Novelty=0,Diversity=0); c=0
    recommended=set()
    logN=np.log2(np.maximum(pop,1)+1)                          # 新颖性用 / for novelty
    for u in range(m):
        rel=test_pos[u]
        if not rel: continue
        rec=topk(fn,u,K); recommended|=set(rec.tolist())
        hits=np.array([1 if i in rel else 0 for i in rec])     # 每个位置是否命中 / hit indicator
        nh=hits.sum()
        acc["Precision"]+=nh/K                                 # 前K命中比例 / precision@K
        acc["Recall"]   +=nh/len(rel)                          # 相关被召回比例 / recall@K
        acc["HitRate"]  +=1.0 if nh>0 else 0.0                 # 至少一个命中 / hit rate
        # MAP@K：每个命中位置的 precision 取平均 / average precision
        hc=0; ap=0.0
        for idx,h in enumerate(hits):
            if h: hc+=1; ap+=hc/(idx+1)
        acc["MAP"]+=ap/min(K,len(rel))
        # MRR：第一个命中的倒数排名 / reciprocal rank of first hit
        nz=np.nonzero(hits)[0]; acc["MRR"]+= 1.0/(nz[0]+1) if len(nz) else 0.0
        # NDCG@K：log 折扣增益 / IDCG / discounted gain normalized
        dcg=np.sum(hits/np.log2(np.arange(2,K+2)))
        idcg=np.sum(1.0/np.log2(np.arange(2,min(K,len(rel))+2)))
        acc["NDCG"]+=dcg/idcg
        # 新颖性：推荐物品越冷门越新颖 / novelty: rarer items = more novel (self-information)
        acc["Novelty"]+= np.mean([np.log2(n/ max(pop[i],1)) for i in rec])
        # 列表内多样性：1 - 平均两两类型相似度 / intra-list diversity
        V=gnorm[rec]; sim=V@V.T; iu=np.triu_indices(len(rec),1)
        acc["Diversity"]+= 1 - sim[iu].mean()
        c+=1
    out={k:v/c for k,v in acc.items()}
    out["Coverage"]=len(recommended)/n                         # 目录覆盖率 / catalog coverage
    return out

rows={name:evaluate(fn,K=10) for name,fn in score.items()}
cols=["Precision","Recall","HitRate","MAP","MRR","NDCG","Coverage","Novelty","Diversity"]
print(f"{'@10':<12}"+"".join(f"{c[:8]:>10}" for c in cols))
for name in score:
    print(f"{name:<12}"+"".join(f"{rows[name][c]:>10.4f}" for c in cols))


**中文**：这张表信息量很大，逐列读：
**English**: This table is dense; read column by column:

**中文**：
1. **准确率类（Precision/Recall/HR/MAP/MRR/NDCG）**：BPR 与 ItemCF **全面碾压 Popularity**，两者互有胜负（BPR 的 Precision/Recall/HR 略高，ItemCF 的 MAP/MRR/NDCG 排序质量略好）。这印证了前面所有章节——个性化模型确实比热门基线强。
2. **覆盖率 Coverage**：**Popularity 只覆盖约 3% 的物品**（给所有人推那几百部爆款），ItemCF 覆盖最广（~43%），BPR 居中。**只看 precision 你会以为 popularity 还行，看 coverage 才暴露它把长尾全饿死了**。
3. **新颖性 Novelty**：Popularity 最低（专推大家都知道的），个性化模型更高——能推出用户没听过但可能喜欢的。
4. **多样性 Diversity**：列表内类型是否丰富，避免"推 10 部几乎一样的电影"。

**English**:
1. **Accuracy metrics (Precision/Recall/HR/MAP/MRR/NDCG)**: BPR and ItemCF **dominate Popularity across the board**, trading wins (BPR slightly higher Precision/Recall/HR; ItemCF slightly better ranking quality on MAP/MRR/NDCG). This confirms all prior chapters — personalized models truly beat the popularity baseline.
2. **Coverage**: **Popularity covers only ~3% of items** (the same few blockbusters for everyone), ItemCF covers the most (~43%), BPR in between. **Precision alone makes popularity look OK; coverage exposes that it starves the long tail.**
3. **Novelty**: Popularity is lowest (recommends what everyone already knows); personalized models surface things users haven't heard of but may like.
4. **Diversity**: whether the list spans genres, avoiding "10 nearly identical movies."


In [ ]:

# ============================================================
# 可视化① 各指标@K 曲线 ② 模型多指标对比 / metric@K curves + multi-metric bars
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(16,4.4))
Ks=[1,5,10,20,50]
ndcg_by_k={name:[evaluate(fn,K)["NDCG"] for K in Ks] for name,fn in score.items()}
rec_by_k ={name:[evaluate(fn,K)["Recall"] for K in Ks] for name,fn in score.items()}
cmap={"Popularity":"#C44E52","ItemCF":"#55A868","BPR":"#4C72B0"}
for name in score:
    ax[0].plot(Ks,ndcg_by_k[name],"o-",label=name,color=cmap[name])
    ax[1].plot(Ks,rec_by_k[name],"o-",label=name,color=cmap[name])
ax[0].set_title("NDCG@K"); ax[0].set_xlabel("K"); ax[0].set_ylabel("NDCG"); ax[0].legend()
ax[1].set_title("Recall@K"); ax[1].set_xlabel("K"); ax[1].set_ylabel("Recall"); ax[1].legend()
# 准确率 vs 覆盖率：散点暴露权衡 / accuracy vs coverage tradeoff
for name in score:
    ax[2].scatter(rows[name]["Coverage"],rows[name]["NDCG"],s=140,color=cmap[name])
    ax[2].annotate(name,(rows[name]["Coverage"],rows[name]["NDCG"]),xytext=(6,4),textcoords="offset points")
ax[2].set_title("准确率 vs 覆盖率 / accuracy vs coverage"); ax[2].set_xlabel("Coverage (越右越广)"); ax[2].set_ylabel("NDCG@10 (越高越准)")
plt.tight_layout(); plt.savefig("/tmp/rec11_viz.png",dpi=80); plt.show()
print("热门基线 NDCG 不算太差，但 Coverage 极低（只服务头部）/ popularity: ok NDCG but tiny coverage")


## 陷阱：采样评估 vs 全量评估 / Pitfall — sampled vs full-ranking evaluation

**中文**：很多论文为了省算力，评估时不对**全部** $N$ 个物品排序，而是"正样本 + 随机采 100 个负样本"里排序（HR@10 在 101 个里算）。**这会严重高估指标，甚至颠倒模型排名**（Krichene & Rendle, KDD 2020）。下面亲手验证：同一批模型，用全量排序 vs 采样排序，看 HR@10 差多少。
**English**: To save compute, many papers rank not against **all** $N$ items but against "the positive + 100 random negatives" (HR@10 among 101). **This massively inflates metrics and can even reorder models** (Krichene & Rendle, KDD 2020). Let's verify hands-on: the same models, full ranking vs sampled, and see how different HR@10 is.


In [ ]:

# ============================================================
# 全量 vs 采样(100负) 评估 HR@10 / full vs sampled (100 negatives)
# ============================================================
rng=np.random.default_rng(0)
def hr_full(fn,K=10):
    h=0;c=0
    for u in range(m):
        if not test_pos[u]:continue
        rec=set(topk(fn,u,K).tolist()); h+=1 if rec&test_pos[u] else 0; c+=1
    return h/c
def hr_sampled(fn,K=10,nneg=100):
    h=0;c=0
    for u in range(m):
        if not test_pos[u]:continue
        sc=fn(u)
        for t in test_pos[u]:                                  # 每个正样本对 100 个随机负样本排序 / 1 pos vs 100 neg
            negs=[]
            while len(negs)<nneg:
                j=rng.integers(n)
                if j not in seen[u] and j not in test_pos[u]: negs.append(j)
            cand=np.array([t]+negs); cs=sc[cand]
            rank=(cs>cs[0]).sum()                              # 正样本在候选里的排名 / rank of the positive
            h+=1 if rank<K else 0; c+=1
    return h/c
print(f"{'模型/model':<12}{'HR@10 全量/full':>16}{'HR@10 采样/sampled':>20}")
for name,fn in score.items():
    print(f"{name:<12}{hr_full(fn):>16.4f}{hr_sampled(fn):>20.4f}")


**中文**：看出问题了吗？**采样评估的 HR@10 比全量高出好几倍**（在 101 个里命中前 10，当然比在 1682 个里容易得多）。更危险的是：采样会**压缩模型间差距、甚至改变排名**，让你在离线得出错误结论。**结论：论文/项目里看到"sampled metrics"要保持警惕，自己评估尽量用全量排序。**
**English**: See the problem? **Sampled HR@10 is several times higher than full ranking** (landing in top-10 of 101 is far easier than of 1682). Worse, sampling **compresses model gaps and can reorder them**, leading to wrong offline conclusions. **Takeaway: be skeptical of "sampled metrics" in papers/projects; use full ranking in your own evaluation.**

> 💼 **实战视角 / Practical angle**
> **中文**：完整评估方法论：① **离线**——全量排序的 NDCG/Recall/MAP + 覆盖率/多样性/新颖性（一篮子指标，别只盯一个）；② **时间切分**而非随机切分（防未来信息泄漏）；③ 永远和 **popularity / 随机** 基线比；④ **线上 A/B**——离线只是筛选，最终看在线 CTR/时长/留存/GMV，并注意离线-在线 gap（位置偏差、反事实问题）；⑤ 关注**业务约束**：新颖性、公平性、不能只优化点击（否则标题党泛滥）。面试金句：*"离线指标用全量排序的 NDCG 为主、辅以覆盖率/多样性；但离线只是代理，决定胜负的是线上 A/B。"*
> **English**: Full methodology: ① **offline** — full-ranking NDCG/Recall/MAP + coverage/diversity/novelty (a basket, not one number); ② **temporal split** not random (avoid future leakage); ③ always compare to **popularity / random** baselines; ④ **online A/B** — offline only filters candidates; what decides is online CTR/dwell/retention/GMV, minding the offline-online gap (position bias, counterfactuals); ⑤ honor **business constraints**: novelty, fairness, don't optimize clicks alone (or clickbait wins). Interview line: *"Offline I lead with full-ranking NDCG plus coverage/diversity; but offline is a proxy — online A/B decides."*

---
### 小结 / Summary
- **中文**：Precision/Recall/HR 不看序，MRR/MAP/NDCG 看序；NDCG 最常用（log 折扣+IDCG 归一化）。
- **English**: Precision/Recall/HR are order-agnostic; MRR/MAP/NDCG are order-aware; NDCG is the standard (log discount + IDCG normalization).
- **中文**：别只看准确率——覆盖率/新颖性/多样性揭示热门基线"饿死长尾"的代价。
- **English**: Don't only look at accuracy — coverage/novelty/diversity expose how the popularity baseline starves the long tail.
- **中文**：两大陷阱：采样评估高估且会乱序(用全量排序)；离线≠在线(最终靠 A/B)。
- **English**: Two pitfalls: sampled eval inflates and reorders (use full ranking); offline ≠ online (A/B decides).

---
**中文**：🎉 至此 **Part 15 · 推荐系统** 全部完成！从基于内容 → 协同过滤 → 矩阵分解 → 隐式反馈 → FM/DeepFM/DCN → Wide&Deep → 双塔召回 → 序列推荐 → 赌博机 → 评估，你已经走通了工业推荐系统的**召回-排序-重排-在线探索-评估**完整链路，并在每一步都用诚实的实验建立了直觉。
**English**: 🎉 **Part 15 · Recommender Systems** is complete! From content-based → collaborative filtering → matrix factorization → implicit feedback → FM/DeepFM/DCN → Wide&Deep → two-tower retrieval → sequential rec → bandits → evaluation, you have walked the full industrial pipeline — **retrieval → ranking → re-ranking → online exploration → evaluation** — building intuition with honest experiments at every step.
